# 12. Paoay: current and combined aligned models

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from experiments.transfer import transfer, transfer_check
from experiments.tuning import tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)


def frame(records):
    table = pd.DataFrame(records)
    for column in ("max_depth", "max_samples"):
        if column in table:
            table[column] = table[column].astype(str)
    return table


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)

## Inputs

In [ ]:
settings_path = root / "settings.local.json"
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
def local_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (root / path).resolve()
kmz_root = local_path(local_settings.get("paoay_kmz_root", "./data/paoay"))
kmz_paths = [kmz_root / name for name in ("00 - Region of Interest.kmz", "00 - Atok Boundary.kmz", "Crop Area.kmz")]
roi_name = local_settings.get("paoay_roi_name", "Region of Interest")
context_names = list(local_settings.get("paoay_context_names", []))
imagery_root = local_path(local_settings["atok_imagery_root"])
cohort = "aligned_full"
print("Composites:", imagery_root)
print("Fixed-window models are not applied: they use May-August and the Paoay season differs.")

In [ ]:
final_check_run = ""  # Optional pipeline_outputs/transfer/final_check_<id>; blank uses the latest
final_check_run = Path(final_check_run) if final_check_run else tuning.latest_run(root, "final_check", area=transfer.AREA)

print("final_check run:", final_check_run)

## Models

In [ ]:
import joblib
models_handoff, current = transfer_check.current_models(root)
saved = joblib.load(final_check_run / "models" / f"{cohort}_combined.joblib")
models = {"current": (current[cohort]["model"], current[cohort]["variant"]),
          "combined": (saved["classifier"], saved["variant"])}
pd.DataFrame([{"model": k, "variant": v, "predictors": len(transfer.variant_columns(v))} for k, (_, v) in models.items()])

## Per-pixel green-up windows

In [ ]:
windows = transfer_check.paoay_windows(kmz_paths, roi_name, context_names, imagery_root)
print("Year:", windows["year"], "| study pixels:", int(windows["mask"].sum()),
      "| pixels with a complete window:", len(windows["selected"]), "| windows not cloud-free:", len(windows["incomplete"]))
display(pd.DataFrame(windows["completeness"]))

## Classify with both models

In [ ]:
run = tuning.new_run(root, "paoay_check", Path(final_check_run / "inputs.json"), area=transfer.AREA,
                     final_check_run=str(final_check_run), year=windows["year"], cohort=cohort)
import rasterio
results, codes = {}, {}
for name, (model, variant) in models.items():
    results[name], codes[name], top = transfer_check.paoay_classify(model, variant, windows)
    profile = {"driver": "GTiff", "width": codes[name].shape[1], "height": codes[name].shape[0],
               "crs": windows["grid"]["crs"], "transform": rasterio.Affine(*windows["grid"]["transform"]), "compress": "deflate"}
    with rasterio.open(run / f"{name}_map.tif", "w", count=1, dtype="uint8", nodata=0, **profile) as raster:
        raster.write(codes[name], 1)
    seasons_rows = [{"row": int(r), "column": int(c), **v} for (r, c), v in sorted(results[name].items())]
    tuning.save_records(run, f"{name}_pixel_predictions", seasons_rows)
    print(name, "classified", len(results[name]), "pixels", flush=True)

## Study-boundary status shares

In [ ]:
status = pd.DataFrame({name: transfer_check.paoay_status(codes[name], windows["mask"]) for name in models})
status.to_csv(run / "status_shares.csv")
display((status * 100).round(1))
confidence = pd.DataFrame([{"model": name, "mean_top_probability": float(np.mean([v["top"] for v in res.values()])),
                            "confident_share_of_predicted": float(np.mean([v["label"] != "Unknown crop type" for v in res.values()]))}
                           for name, res in results.items()])
display(confidence.round(3))

## Agreement between the two maps

In [ ]:
agreement = transfer_check.paoay_agreement(results["current"], results["combined"])
print("Pixels predicted by both:", agreement["pixels"], "| same final label:", round(agreement["same_label_share"], 3))
display(pd.DataFrame(agreement["table"]).pivot(index="current", columns="combined", values="pixels").fillna(0).astype(int))

## Reference areas

In [ ]:
from crop_pipeline.paoay import aligned
area_rows = []
for name in models:
    for row in aligned.area_table(codes[name], windows["mask"], windows["areas"], windows["grid"]):
        area_rows.append({"model": name, **row})
areas = pd.DataFrame(area_rows)
areas.to_csv(run / "reference_areas.csv", index=False)
display(areas)

## Reference area notes

In [ ]:
from crop_pipeline.paoay import references
records, _ = references.read_inputs(kmz_paths)
notes = pd.DataFrame([{"area": r["name"], "notes": (r.get("notes") or "").replace("\n", " ")[:220]}
                      for r in records if r["name"] in set(areas.area)])
display(notes)

## Figure

In [ ]:
from matplotlib.colors import ListedColormap
names = {code: name for name, code in aligned.CODES.items()}
colors = ["white", "#e67e22", "#27ae60", "#8e44ad", "#bdc3c7", "#ecf0f1", "#95a5a6"]
figure, axes = plt.subplots(1, 2, figsize=(14, 7))
for axis, name in zip(axes, models):
    axis.imshow(codes[name], cmap=ListedColormap(colors[:max(names) + 1]), vmin=0, vmax=max(names), interpolation="nearest")
    axis.set_title(name)
    axis.axis("off")
handles = [plt.Rectangle((0, 0), 1, 1, color=colors[c]) for c in sorted(names)]
figure.legend(handles, [names[c] for c in sorted(names)], loc="lower center", ncol=4)
show_and_save(figure, run, "paoay_maps")
print("Run folder:", run)